# Qwen3-8B TT-matrix 微调前后 WikiText 对比

本 Notebook 只负责配置和分阶段调用。缓存签名、WikiText 分块、断点恢复、TT-only 微调和结果汇总均由 `qwen3_tn.wikitext_workflow` 实现。

流程使用同一模型实例比较 Dense Baseline、替换 `[0, 6)` 层 `down_proj` 后的 TT 模型、以及微调 TT cores 一轮后的模型。评测和训练产物都带严格配置签名；相同配置重复 Run All 会读取已有 JSON/final checkpoint，中断后会从最近的合法 `checkpoint-N` 恢复。

In [14]:
import gc
import os
import sys
from dataclasses import asdict
from pathlib import Path
from pprint import pprint

os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("HF_DATASETS_OFFLINE", "1")

PROJECT_ROOT = Path("/home/xls/workspace/projects/qwen3-tn-compression").resolve()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

ipython = get_ipython()
ipython.run_line_magic("load_ext", "autoreload")
ipython.run_line_magic("autoreload", "2")

old_workflow = globals().get("workflow")
if old_workflow is not None:
    old_workflow.close()
globals().pop("workflow", None)
gc.collect()

from qwen3_tn import (
    TTFineTuneConfig,
    WikiTextTTFineTuneWorkflow,
    WikiTextTTWorkflowConfig,
    build_qwen_mlp_tt_matrix_targets,
    print_comparison,
)
from qwen3_tn.model_evaluation import EvaluationConfig

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. 实验配置

`TT_BACKEND` 可设为 `native` 或 `tensorly_torch`；后者需先在终端执行 `pip install -e ".[tensorly]"`。Dense baseline 共用，TT 评测和训练产物按 backend 分目录保存。`QUICK_RUN=True` 只训练 8 个 blocks、评测 1 个样本，用于检查链路。正式实验保持 `False`。每个训练 checkpoint 的 `trainer_state.json` 都包含完整训练参数和训练签名 SHA-256。

In [15]:
MODEL_PATH = Path("/infini-data/Qwen3-8B")
DEVICE = "cuda:0"
LAYER_INDICES = list(range(0, 6))
PROJECTION_CONFIGS = {
    "down_proj": {
        "out_modes": (8, 8, 8, 8),
        "in_modes": (8, 8, 8, 24),
        "ranks": (1, 64, 2048, 192, 1),
        "token_chunk_size": 8,
    }
}
LAYER_OVERRIDES = {}

TT_BACKEND = "native"  # 可改为 "tensorly_torch"
QUICK_RUN = True
FORCE_RETRAIN = False
FORCE_REEVALUATE = False
FORCE_RECOMPUTE = False
QUICK_TRAIN_BLOCKS = 8

EVALUATION_CONFIG_PATH = PROJECT_ROOT / "configs/evaluation/wikitext.json"
evaluation_config = EvaluationConfig.from_json(
    EVALUATION_CONFIG_PATH,
    **({"limit": 1} if QUICK_RUN else {}),
)
training_config = TTFineTuneConfig(
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-5,
    weight_decay=0.0,
    warmup_ratio=0.03,
    max_grad_norm=1.0,
    max_length=1024,
    logging_steps=10,
    save_steps=10,
    save_total_limit=2,
    seed=42,
    device=DEVICE,
    bf16_autocast=True,
    gradient_checkpointing=True,
    tt_activation_checkpointing=True,
)

run_suffix = "_quick" if QUICK_RUN else ""
layer_label = f"{LAYER_INDICES[0]}_{LAYER_INDICES[-1]}"
workflow_config = WikiTextTTWorkflowConfig(
    model_path=MODEL_PATH,
    result_root=PROJECT_ROOT / "results" / f"wikitext_finetune_layer_{layer_label}{run_suffix}",
    artifact_root=PROJECT_ROOT / "artifacts" / f"tt_finetune_wikitext_layer_{layer_label}{run_suffix}",
    decomposition_cache_root=PROJECT_ROOT / "artifacts/tt_matrix_cache",
    evaluation=evaluation_config,
    training=training_config,
    device=DEVICE,
    tt_backend=TT_BACKEND,
    quick_run=QUICK_RUN,
    quick_train_blocks=QUICK_TRAIN_BLOCKS,
    force_retrain=FORCE_RETRAIN,
    force_reevaluate=FORCE_REEVALUATE,
    force_recompute=FORCE_RECOMPUTE,
    svd_driver="gesvd",
    min_free_gpu_gib=20,
)
targets = build_qwen_mlp_tt_matrix_targets(
    LAYER_INDICES, PROJECTION_CONFIGS, LAYER_OVERRIDES
)
workflow = WikiTextTTFineTuneWorkflow(workflow_config, targets)

print("模式：", workflow_config.mode.upper())
print("TT backend：", workflow_config.tt_backend)
print("Baseline 文件：", workflow_config.baseline_result_path)
print("TT 评测结果目录：", workflow_config.backend_result_root)
print("TT 训练产物目录：", workflow_config.backend_artifact_root)
pprint(evaluation_config)
pprint(training_config)

模式： QUICK
评测结果目录： /mnt/intern7/xls/projects/qwen3-tn-compression/results/wikitext_finetune_layer_0_5_quick
训练产物目录： /mnt/intern7/xls/projects/qwen3-tn-compression/artifacts/tt_finetune_wikitext_layer_0_5_quick
EvaluationConfig(task='wikitext',
                 limit=1,
                 batch_size=1,
                 max_length=2048,
                 num_fewshot=0,
                 apply_chat_template=False,
                 bootstrap_iters=0,
                 seed=42)
TTFineTuneConfig(num_train_epochs=1,
                 max_steps=None,
                 per_device_train_batch_size=1,
                 gradient_accumulation_steps=8,
                 learning_rate=1e-05,
                 weight_decay=0.0,
                 warmup_ratio=0.03,
                 max_grad_norm=1.0,
                 max_length=1024,
                 logging_steps=10,
                 save_steps=10,
                 save_total_limit=2,
                 seed=42,
                 device='cuda:0',
                 bf

## 2. 加载固定 backbone，并获取 Dense Baseline

完整 Qwen backbone 只加载一次。Baseline JSON 的模型签名和评测配置完全匹配时不会再次调用 lm-eval。

In [16]:
workflow.load()
baseline = workflow.evaluate_baseline()
print("Baseline 来源：", baseline.source)

Loading checkpoint shards: 100%|██████████| 5/5 [00:00<00:00, 124.37it/s]


loaded evaluation cache: /mnt/intern7/xls/projects/qwen3-tn-compression/results/wikitext_finetune_layer_0_5_quick/baseline.json
Baseline 来源： cache


## 3. 安装 TT 层并评测微调前结果

分解优先读取逐层 TT-SVD cache。工作流只安装 `targets` 中声明的层，并为初始 cores 生成严格签名。

In [17]:
tt_before = workflow.install_tt_and_evaluate_before()
print("TT 微调前结果来源：", tt_before.source)
print("TT 分解汇总：")
pprint(workflow.decomposition["aggregate"])

[1/6] model.layers.0.mlp.down_proj：检查缓存元数据
[1/6] model.layers.0.mlp.down_proj：元数据命中，验证权重 SHA-256


[1/6] model.layers.0.mlp.down_proj：缓存命中，SHA-256 已验证，0.00 秒加载完成
[2/6] model.layers.1.mlp.down_proj：检查缓存元数据
[2/6] model.layers.1.mlp.down_proj：元数据命中，验证权重 SHA-256
[2/6] model.layers.1.mlp.down_proj：缓存命中，SHA-256 已验证，0.00 秒加载完成
[3/6] model.layers.2.mlp.down_proj：检查缓存元数据
[3/6] model.layers.2.mlp.down_proj：元数据命中，验证权重 SHA-256
[3/6] model.layers.2.mlp.down_proj：缓存命中，SHA-256 已验证，0.00 秒加载完成
[4/6] model.layers.3.mlp.down_proj：检查缓存元数据
[4/6] model.layers.3.mlp.down_proj：元数据命中，验证权重 SHA-256
[4/6] model.layers.3.mlp.down_proj：缓存命中，SHA-256 已验证，0.00 秒加载完成
[5/6] model.layers.4.mlp.down_proj：检查缓存元数据
[5/6] model.layers.4.mlp.down_proj：元数据命中，验证权重 SHA-256
[5/6] model.layers.4.mlp.down_proj：缓存命中，SHA-256 已验证，0.00 秒加载完成
[6/6] model.layers.5.mlp.down_proj：检查缓存元数据
[6/6] model.layers.5.mlp.down_proj：元数据命中，验证权重 SHA-256
[6/6] model.layers.5.mlp.down_proj：缓存命中，SHA-256 已验证，0.00 秒加载完成


`pretrained` model kwarg is not of type `str`. Many other model arguments may be ignored. Please do not launch via accelerate or use `parallelize=True` if passing an existing model this way.
Passed an already-initialized model through `pretrained`, assuming single-process call to evaluate() or custom distributed integration


evaluation cache signature mismatch; reevaluating: /mnt/intern7/xls/projects/qwen3-tn-compression/results/wikitext_finetune_layer_0_5_quick/tt_before_finetune.json
开始评测任务：wikitext


[Task: wikitext] metric word_perplexity is defined, but aggregation is not. using default aggregation=weighted_perplexity
[Task: wikitext] metric word_perplexity is defined, but higher_is_better is not. using default higher_is_better=False
[Task: wikitext] metric byte_perplexity is defined, but aggregation is not. using default aggregation=weighted_perplexity
[Task: wikitext] metric byte_perplexity is defined, but higher_is_better is not. using default higher_is_better=False
[Task: wikitext] metric bits_per_byte is defined, but aggregation is not. using default aggregation=bits_per_byte
[Task: wikitext] metric bits_per_byte is defined, but higher_is_better is not. using default higher_is_better=False
Using the latest cached version of the dataset since EleutherAI/wikitext_document_level couldn't be found on the Hugging Face Hub (offline mode is enabled).
Found the latest cached dataset configuration 'wikitext-2-raw-v1' at /home/xls/.cache/huggingface/datasets/EleutherAI___wikitext_docu

任务评测完成：wikitext，耗时 11.73 秒
完整结果已保存到：/mnt/intern7/xls/projects/qwen3-tn-compression/results/wikitext_finetune_layer_0_5_quick/tt_before_finetune.json
TT 微调前结果来源： evaluation
TT 分解汇总：
{'cache_hit_count': 6,
 'cache_load_seconds': 0.01842236891388893,
 'compressed_model_parameters': 8090317824,
 'decomposed_count': 0,
 'dense_target_parameters': 301989888,
 'matrix_error_frobenius_norm_squared': 35802.91542947695,
 'matrix_reference_frobenius_norm_squared': 138687.19733680828,
 'matrix_weight_all_finite': True,
 'matrix_weight_relative_l2': 0.5080904247125808,
 'model_compression_ratio': 1.012412063182748,
 'model_parameter_reduction': 100417536,
 'model_parameter_reduction_fraction': 0.012259892620923307,
 'original_model_parameters': 8190735360,
 'target_compression_ratio': 1.4981711777615216,
 'target_count': 6,
 'target_parameter_reduction': 100417536,
 'target_parameter_reduction_fraction': 0.33251953125,
 'tt_matrix_target_parameters': 201572352,
 'weight_fingerprint_seconds': 0.7086

## 4. 准备 WikiText 训练数据

文档之间插入 EOS，连接后按 1024 tokens 分块。数据 fingerprint、token/block 数、TT ranks 和全部训练参数共同决定训练签名。

In [18]:
training_data = workflow.prepare_training_data()
pprint(asdict(training_data))

Using the latest cached version of the dataset since EleutherAI/wikitext_document_level couldn't be found on the Hugging Face Hub (offline mode is enabled).
Found the latest cached dataset configuration 'wikitext-2-raw-v1' at /home/xls/.cache/huggingface/datasets/EleutherAI___wikitext_document_level/wikitext-2-raw-v1/0.0.0/647234772b9554e208af6c826f23b99e3cac88c8 (last modified on Mon Aug 24 15:08:36 2026).


{'blocks': 8,
 'dataset_fingerprint': '35de03d316a46c28',
 'documents': 629,
 'full_token_count': 2519154,
 'optimizer_updates': 1,
 'selected_token_count': 8192,
 'training_signature_sha256': 'b63f69f2ed72cabde75644ec222e03f1c427130a56c3124ad82d36b24e100719'}


## 5. 微调或恢复 TT cores

只训练 TT cores 一轮。每 10 个 optimizer updates 落盘一次并保留最近 2 个；Notebook 关闭后重新 Run All，会在签名完全一致时从最近的合法 checkpoint 继续。

In [19]:
training = workflow.train_or_load()
print("训练结果来源：", training.source)
print("训练后 core 签名：", training.core_signature["sha256"])
peak_bytes = training.metrics.get("peak_cuda_memory_bytes")
if peak_bytes is not None:
    print(f"训练峰值 CUDA 显存：{peak_bytes / 1024**3:.2f} GiB")
pprint(training.metrics)

training signature mismatch; fine-tuning again
training only TT cores: {'module_paths': ('model.layers.0.mlp.down_proj', 'model.layers.1.mlp.down_proj', 'model.layers.2.mlp.down_proj', 'model.layers.3.mlp.down_proj', 'model.layers.4.mlp.down_proj', 'model.layers.5.mlp.down_proj'), 'parameter_tensors': 24, 'trainable_parameters': 201572352, 'total_parameters': 8090317824}


You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


训练结果来源： training
训练后 core 签名： 49a27415a7a5195159901957b01863f992e12339a048285646a8fcc352f772d2
训练峰值 CUDA 显存：68.59 GiB
{'global_step': 1,
 'last_training_loss': 2.5788817405700684,
 'mean_training_loss': 2.679276406764984,
 'model_gradient_checkpointing': True,
 'peak_cuda_memory_bytes': 73643805696,
 'resume_checkpoint': '/mnt/intern7/xls/projects/qwen3-tn-compression/artifacts/tt_finetune_wikitext_layer_0_5_quick/checkpoint-1',
 'resumed_from': None,
 'total_model_parameters': 8090317824,
 'trainable_tt_parameters': 201572352,
 'tt_activation_checkpointing': True,
 'tt_module_paths': ['model.layers.0.mlp.down_proj',
                     'model.layers.1.mlp.down_proj',
                     'model.layers.2.mlp.down_proj',
                     'model.layers.3.mlp.down_proj',
                     'model.layers.4.mlp.down_proj',
                     'model.layers.5.mlp.down_proj']}


## 6. 微调后评测与差异汇总

后测使用导出的 BF16 cores。三个评测结果分别缓存，比较结果写入 `comparison.json`。

In [20]:
comparison = workflow.evaluate_after_and_compare()
print("TT 微调后结果来源：", comparison.evaluation.source)
print_comparison(comparison.comparison)
print("比较汇总已保存：", workflow_config.comparison_path)

`pretrained` model kwarg is not of type `str`. Many other model arguments may be ignored. Please do not launch via accelerate or use `parallelize=True` if passing an existing model this way.
Passed an already-initialized model through `pretrained`, assuming single-process call to evaluate() or custom distributed integration


evaluation cache signature mismatch; reevaluating: /mnt/intern7/xls/projects/qwen3-tn-compression/results/wikitext_finetune_layer_0_5_quick/tt_after_finetune.json
开始评测任务：wikitext


[Task: wikitext] metric word_perplexity is defined, but aggregation is not. using default aggregation=weighted_perplexity
[Task: wikitext] metric word_perplexity is defined, but higher_is_better is not. using default higher_is_better=False
[Task: wikitext] metric byte_perplexity is defined, but aggregation is not. using default aggregation=weighted_perplexity
[Task: wikitext] metric byte_perplexity is defined, but higher_is_better is not. using default higher_is_better=False
[Task: wikitext] metric bits_per_byte is defined, but aggregation is not. using default aggregation=bits_per_byte
[Task: wikitext] metric bits_per_byte is defined, but higher_is_better is not. using default higher_is_better=False
Using the latest cached version of the dataset since EleutherAI/wikitext_document_level couldn't be found on the Hugging Face Hub (offline mode is enabled).
Found the latest cached dataset configuration 'wikitext-2-raw-v1' at /home/xls/.cache/huggingface/datasets/EleutherAI___wikitext_docu

任务评测完成：wikitext，耗时 11.83 秒
完整结果已保存到：/mnt/intern7/xls/projects/qwen3-tn-compression/results/wikitext_finetune_layer_0_5_quick/tt_after_finetune.json
TT 微调后结果来源： evaluation
model                    metric                                  value       source
------------------------------------------------------------------------------------
baseline                 word_perplexity,none                 9.254118        cache
baseline                 bits_per_byte,none                   0.619134        cache
tt_before_finetune       word_perplexity,none                 9.254118   evaluation
tt_before_finetune       bits_per_byte,none                   0.619134   evaluation
tt_after_finetune        word_perplexity,none                 8.987111   evaluation
tt_after_finetune        bits_per_byte,none                   0.610988   evaluation

deltas (percentage columns use %):
metric                            After-Before   After/Before %   Before-Baseline   After-Baseline  recovered %
--------

## 7. 清理

恢复 Dense 层并释放模型。如果前面某个 cell 异常中断，也可以单独运行本 cell。

In [21]:
workflow = globals().get("workflow")
if workflow is not None:
    workflow.close()
globals().pop("workflow", None)
gc.collect()
print("清理完成")

清理完成
